# 05 - Benchmark 脚本深度剖析与定制

## 学习目标
- 逐行理解 `bench_sglang_eagle.py` 的完整逻辑
- 掌握 `bench_sglang_chat.py` 的高级特性（HTTP 并发、MTP Head 分析）
- 理解评测指标体系的完整定义
- 实战：编写自定义 benchmark

## 1. bench_sglang_eagle.py 源码解析

**文件**: `evaluation_dev/aiak_bench/bench_sglang_eagle.py`

这是 qf_mtp_eval 中最基础的 benchmark 脚本，使用 SGLang Python SDK 进行批量推理评测。

### 整体流程

```
main(args)
  │
  ├── 1. load_questions(filename, mt_mode) → questions[]
  │       支持多种格式 (messages / request.body / instruction+history)
  │
  ├── 2. detect_turn_mode(questions) → (mode, single_count, multi_count)
  │       判断单轮/多轮/混合模式
  │
  ├── 3. build arguments[] — 构建 sgl.function 的输入
  │       处理 system message / thinking token
  │
  ├── 4. select_sglang_backend(args) → backend
  │       连接到 SGLang server
  │
  ├── 5. answer_custom_bench.run_batch(arguments, ...)
  │       核心批量推理调用
  │
  ├── 6. collect metrics — 从 meta_info 中提取统计
  │       ├── completion_tokens, prompt_tokens, cached_tokens
  │       ├── spec_verify_ct → accept_length
  │       └── throughput = total_tokens / latency
  │
  ├── 7. write_answers(filename, ...) → answers.jsonl
  │
  └── 8. write result.json → 汇总指标
```

In [ ]:
import json
import os
import time
import uuid

# ============================================================
# 1. load_questions — 数据加载（支持多格式）
# ============================================================

def extract_text_content(content):
    """Extract text from content that may be string or list of content parts.
    
    Handles:
      - "plain string"
      - [{"type": "text", "text": "..."}, {"type": "image_url", ...}]
    """
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
            elif isinstance(item, str):
                parts.append(item)
        return "\n".join(parts)
    return str(content)


def count_user_turns(messages):
    """Count user turns in a conversation."""
    return sum(1 for m in messages if m.get("role", "user") == "user")


def load_questions(filename, mt_mode="all", num_questions=None):
    """Load questions from JSONL, supporting multiple formats.
    
    Formats supported:
    1. {"messages": [...], "system": "..."} — standard
    2. {"request": {"body": {"messages": [...]}}} — nested API format
    3. {"instruction": "...", "history": [...]} — legacy format
    
    Args:
        mt_mode: "single" (1 user turn), "multi" (>1), "all" (no filter)
    """
    questions = []
    with open(filename, "r") as f:
        for i, line in enumerate(f):
            if num_questions and i >= num_questions:
                break
            obj = json.loads(line)
            
            messages = None
            system = ""
            answer = obj.get("answer")  # ground truth (for c_eval etc.)
            
            # Format 1: direct messages
            if "messages" in obj:
                messages = obj["messages"]
                system = obj.get("system", "")
            # Format 2: nested in request.body
            elif "request" in obj and "body" in obj["request"]:
                messages = obj["request"]["body"].get("messages", [])
                system = obj["request"]["body"].get("system", "")
            # Format 3: instruction + history
            elif "instruction" in obj:
                instruction = obj["instruction"]
                system = obj.get("system", "")
                history = obj.get("history", [])
                messages = []
                for turn in history:
                    if isinstance(turn, list) and len(turn) >= 2:
                        if turn[0]: messages.append({"role": "user", "content": turn[0]})
                        if turn[1]: messages.append({"role": "assistant", "content": turn[1]})
                messages.append({"role": "user", "content": instruction})
            
            if not messages:
                continue
            
            # Filter by turn mode
            turns = count_user_turns(messages)
            if mt_mode == "single" and turns > 1:
                continue
            if mt_mode == "multi" and turns <= 1:
                continue
            
            question_data = {"id": len(questions), "messages": messages, "system": system}
            if answer is not None:
                question_data["answer"] = answer
            questions.append(question_data)
    
    return questions


# 测试
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")
QUESTION_FILE = os.path.join(DATA_DIR, "sample_questions.jsonl")

all_q = load_questions(QUESTION_FILE, mt_mode="all")
single_q = load_questions(QUESTION_FILE, mt_mode="single")
multi_q = load_questions(QUESTION_FILE, mt_mode="multi")

print(f"All questions:    {len(all_q)}")
print(f"Single-turn:      {len(single_q)}")
print(f"Multi-turn:       {len(multi_q)}")

In [ ]:
# ============================================================
# 2. detect_turn_mode — 自动检测数据集的对话模式
# ============================================================

def detect_turn_mode(questions):
    """Detect turn mode: single, multi, or mixed.
    
    Rule: if 90%+ is one type, classify as that type.
    """
    single_count = sum(1 for q in questions if count_user_turns(q["messages"]) == 1)
    multi_count = len(questions) - single_count
    total = len(questions)
    
    if total == 0:
        return "unknown", 0, 0
    
    single_ratio = single_count / total
    if single_ratio >= 0.9:
        mode = "single"
    elif single_ratio <= 0.1:
        mode = "multi"
    else:
        mode = "mixed"
    
    return mode, single_count, multi_count


mode, sc, mc = detect_turn_mode(all_q)
print(f"Dataset mode: {mode}")
print(f"Single-turn: {sc}, Multi-turn: {mc}")

In [ ]:
# ============================================================
# 3. @sgl.function — 核心生成函数
# ============================================================
import sglang as sgl

THINK_TOKEN = '<think>'

@sgl.function
def answer_custom_bench(s, conv_messages):
    """The core generation function used in bench_sglang_eagle.py.
    
    Handles full conversation: system, then alternating user/assistant turns,
    with the last user turn triggering generation.
    """
    for msg in conv_messages:
        role = msg["role"]
        content = msg["content"]
        if role == "system":
            s += sgl.system(content)
        elif role == "user":
            s += sgl.user(content)
        elif role == "assistant":
            s += sgl.assistant(content)
    # Generate the final assistant response
    s += sgl.assistant(sgl.gen("answer"))


def build_arguments(questions, thinking=False):
    """Build run_batch arguments from loaded questions.
    
    If thinking=True, append <think> token to the last user message
    to trigger chain-of-thought generation.
    """
    arguments = []
    valid_questions = []
    
    for q in questions:
        conv_messages = []
        system = q.get("system", "")
        if system:
            conv_messages.append({"role": "system", "content": system})
        
        for msg in q["messages"]:
            role = msg.get("role", "user")
            if role == "system":
                if not system:
                    conv_messages.append({"role": "system", "content": extract_text_content(msg["content"])})
                continue
            conv_messages.append({"role": role, "content": extract_text_content(msg["content"])})
        
        # Append think token for reasoning mode
        if thinking:
            for i in range(len(conv_messages) - 1, -1, -1):
                if conv_messages[i]["role"] == "user":
                    conv_messages[i]["content"] += THINK_TOKEN
                    break
        
        if conv_messages:
            arguments.append({"conv_messages": conv_messages})
            valid_questions.append(q)
    
    return arguments, valid_questions


# Build arguments
args_normal, valid_q = build_arguments(all_q, thinking=False)
args_think, _ = build_arguments(all_q[:2], thinking=True)

print(f"Normal args count: {len(args_normal)}")
print(f"\nThinking mode example (last user msg has <think>):")
print(f"  {args_think[0]['conv_messages'][-1]['content'][-50:]}")

## 2. bench_sglang_chat.py — HTTP 并发评测

**文件**: `evaluation_dev/common/bench_sglang_chat.py`

这是更高级的 benchmark 脚本，使用 HTTP API 而非 SDK，支持：
- `ThreadPoolExecutor` 并发 HTTP 请求
- 流式输出 (streaming)
- MTP Head 逐头接受率分析
- 实时进度写入

### 与 eagle 版本的区别

| 方面 | bench_sglang_eagle.py | bench_sglang_chat.py |
|------|----------------------|---------------------|
| 调用方式 | SGLang SDK (`sgl.function`) | HTTP API (requests) |
| 并发方式 | `run_batch(num_threads)` | `ThreadPoolExecutor` |
| 兼容性 | 仅 SGLang backend | 任何 OpenAI 兼容 API |
| MTP 分析 | 仅 accept_length | 逐 MTP Head 分析 |
| 流式支持 | 不支持 | 支持 SSE streaming |

In [ ]:
# ============================================================
# bench_sglang_chat.py 的核心逻辑复现
# ============================================================
import concurrent.futures
import urllib.request
import urllib.error


def send_chat_request(url, messages, temperature=0, max_tokens=512, timeout=120):
    """Send a single chat completion request via HTTP.
    
    This is how bench_sglang_chat.py sends requests.
    """
    payload = {
        "model": "default",
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False,
    }
    
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=data,
        headers={"Content-Type": "application/json"},
    )
    
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            result = json.loads(resp.read().decode())
            return result
    except (urllib.error.URLError, TimeoutError) as e:
        return {"error": str(e)}


def run_chat_benchmark(host, port, questions, max_tokens=512, num_threads=8):
    """Run benchmark using HTTP chat/completions API with concurrent requests."""
    url = f"http://{host}:{port}/v1/chat/completions"
    results = [None] * len(questions)
    
    def process_one(idx):
        q = questions[idx]
        resp = send_chat_request(url, q["messages"], max_tokens=max_tokens)
        results[idx] = resp
        return idx
    
    tic = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(process_one, i) for i in range(len(questions))]
        for f in concurrent.futures.as_completed(futures):
            f.result()  # raises if error
    latency = time.perf_counter() - tic
    
    return results, latency


print("Chat benchmark functions loaded.")
print("Note: This uses HTTP API instead of SGLang SDK.")

### MTP Head 逐头分析

在 `bench_sglang_chat.py` 中，当模型有多个 MTP head 时，可以分析每个 head 的接受率。

```
MTP Head Analysis (4 heads):
  Head 0 (next token):    accept_rate = 0.95  ← 几乎总是接受
  Head 1 (token +2):      accept_rate = 0.78  ← 较高
  Head 2 (token +3):      accept_rate = 0.52  ← 中等
  Head 3 (token +4):      accept_rate = 0.31  ← 较低但仍有贡献
  
  Cumulative rates:
  Head 0:   0.95 (at least 1 token accepted)
  Head 0-1: 0.74 (at least 2 tokens accepted)
  Head 0-2: 0.39 (at least 3 tokens accepted)
  Head 0-3: 0.12 (all 4 tokens accepted)
```

In [ ]:
import numpy as np

# ============================================================
# MTP Head Accept Rate 计算
# ============================================================

def compute_mtp_head_accept_rates(meta_infos, num_heads=4):
    """Compute per-head accept rates from meta_info.
    
    In qf_mtp_eval, this is tracked as:
    - macro_mtp_head_accept_rates: averaged per sample
    - micro_mtp_head_accept_rates: aggregated across all samples
    - macro_mtp_head_cumulative_accept_rates: cumulative acceptance
    """
    # Simulated data: for each sample, acceptance result at each head position
    # In real code, this comes from detailed server response
    np.random.seed(42)
    
    # Simulate: each head has decreasing probability of acceptance
    base_rates = [0.95, 0.78, 0.52, 0.31]  # per-head rates
    num_samples = len(meta_infos)
    
    per_head_accepts = np.zeros((num_samples, num_heads))
    for i in range(num_samples):
        for h in range(num_heads):
            per_head_accepts[i, h] = np.random.binomial(1, base_rates[h])
    
    # Macro: average per-head rate across samples
    macro_rates = per_head_accepts.mean(axis=0)
    
    # Cumulative: probability that ALL heads up to h accept
    cumulative = np.cumprod(per_head_accepts, axis=1).mean(axis=0)
    
    return {
        "per_head_accept_rates": macro_rates.tolist(),
        "cumulative_accept_rates": cumulative.tolist(),
    }


# Demo with simulated data
demo_metas = [{"completion_tokens": 100 + i*50, "spec_verify_ct": 30 + i*15} for i in range(20)]
head_analysis = compute_mtp_head_accept_rates(demo_metas, num_heads=4)

print("=== MTP Head Analysis ===")
print(f"\nPer-head accept rates:")
for h, rate in enumerate(head_analysis["per_head_accept_rates"]):
    print(f"  Head {h}: {rate:.3f}")

print(f"\nCumulative accept rates:")
for h, rate in enumerate(head_analysis["cumulative_accept_rates"]):
    print(f"  Heads 0-{h}: {rate:.3f} (at least {h+1} tokens accepted)")

# Theoretical accept length from cumulative rates
theoretical_al = 1 + sum(head_analysis["cumulative_accept_rates"])
print(f"\nTheoretical Accept Length: {theoretical_al:.3f}")

## 3. 完整评测指标体系

在 `qf_mtp_eval` 的 `web_service.py` 中定义了所有追踪的指标：

In [ ]:
# qf_mtp_eval 追踪的完整指标体系

METRICS_DEFINITION = {
    # === 核心性能指标 ===
    "throughput": {
        "formula": "sum(completion_tokens) / total_latency",
        "unit": "tokens/s",
        "description": "System output throughput",
    },
    "latency": {
        "formula": "end_time - start_time",
        "unit": "seconds",
        "description": "Total wall-clock time for all requests",
    },
    
    # === Accept Length 变体 ===
    "macro_accept_length": {
        "formula": "mean(per_sample_AL)",
        "description": "Average of per-sample accept lengths (equal weight)",
    },
    "micro_accept_length": {
        "formula": "sum(all_completion_tokens) / sum(all_spec_verify_ct)",
        "description": "Global tokens/verify ratio (long samples weigh more)",
    },
    "pooled_accept_length": {
        "formula": "same as micro (pooled across all)",
        "description": "Alias for micro accept length",
    },
    "filtered_accept_length": {
        "formula": "macro_AL after filtering short samples",
        "description": "Excludes very short outputs that skew AL",
    },
    
    # === MTP Head 分析 ===
    "macro_mtp_head_accept_rates": {
        "formula": "mean(per_sample_per_head_accept_rate)",
        "description": "Per-MTP-head acceptance rate (macro averaged)",
    },
    "micro_mtp_head_accept_rates": {
        "formula": "sum(per_head_accepted) / sum(per_head_total)",
        "description": "Per-head acceptance rate (micro averaged)",
    },
    "macro_mtp_head_cumulative_accept_rates": {
        "formula": "P(heads 0..h all accept)",
        "description": "Cumulative probability of accepting h+1 consecutive tokens",
    },
}

print("=== qf_mtp_eval Metrics System ===")
for name, info in METRICS_DEFINITION.items():
    print(f"\n{name}:")
    print(f"  Formula: {info['formula']}")
    print(f"  Description: {info['description']}")

## 4. 实战：编写自定义 Benchmark

现在我们综合以上知识，编写一个完整的自定义 benchmark。

In [ ]:
# ============================================================
# 自定义 Benchmark：完整实现
# ============================================================

class MTPBenchmark:
    """A complete MTP evaluation benchmark.
    
    Usage:
        bench = MTPBenchmark(host="127.0.0.1", port=30000)
        result = bench.run(question_file, num_questions=100)
        bench.save_results(result, answer_file, result_file)
    """
    
    def __init__(self, host="127.0.0.1", port=30000):
        self.host = host
        self.port = port
        self.url = f"http://{host}:{port}/v1/chat/completions"
    
    def run(self, question_file, num_questions=None, max_tokens=512, 
            num_threads=8, mt_mode="all", thinking=False):
        """Run the full benchmark pipeline."""
        # 1. Load questions
        questions = load_questions(question_file, mt_mode=mt_mode, num_questions=num_questions)
        mode, sc, mc = detect_turn_mode(questions)
        print(f"Loaded {len(questions)} questions (mode={mode}, single={sc}, multi={mc})")
        
        # 2. Build request payloads
        payloads = self._build_payloads(questions, thinking)
        
        # 3. Execute requests concurrently
        print(f"Running {len(payloads)} requests with {num_threads} threads...")
        responses, latency = self._execute_batch(payloads, max_tokens, num_threads)
        
        # 4. Compute metrics
        metrics = self._compute_metrics(responses, latency)
        metrics["num_requests"] = len(questions)
        metrics["single_turn_count"] = sc
        metrics["multi_turn_count"] = mc
        metrics["mode"] = mode
        
        return {
            "metrics": metrics,
            "questions": questions,
            "responses": responses,
        }
    
    def _build_payloads(self, questions, thinking):
        """Build HTTP request payloads."""
        payloads = []
        for q in questions:
            messages = []
            system = q.get("system", "")
            if system:
                messages.append({"role": "system", "content": system})
            for msg in q["messages"]:
                role = msg.get("role", "user")
                if role != "system":
                    messages.append({"role": role, "content": extract_text_content(msg["content"])})
            
            if thinking and messages:
                for i in range(len(messages) - 1, -1, -1):
                    if messages[i]["role"] == "user":
                        messages[i]["content"] += THINK_TOKEN
                        break
            
            payloads.append(messages)
        return payloads
    
    def _execute_batch(self, payloads, max_tokens, num_threads):
        """Execute requests using ThreadPoolExecutor."""
        responses = [None] * len(payloads)
        
        def do_request(idx):
            resp = send_chat_request(self.url, payloads[idx], max_tokens=max_tokens)
            responses[idx] = resp
        
        tic = time.perf_counter()
        with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as ex:
            futures = [ex.submit(do_request, i) for i in range(len(payloads))]
            for f in concurrent.futures.as_completed(futures):
                f.result()
        latency = time.perf_counter() - tic
        
        return responses, latency
    
    def _compute_metrics(self, responses, latency):
        """Compute all MTP evaluation metrics."""
        total_completion_tokens = 0
        total_prompt_tokens = 0
        total_verify_ct = 0
        sample_als = []
        has_spec = False
        
        for resp in responses:
            if "error" in resp or "usage" not in resp:
                continue
            
            usage = resp["usage"]
            comp = usage.get("completion_tokens", 0)
            prompt = usage.get("prompt_tokens", 0)
            total_completion_tokens += comp
            total_prompt_tokens += prompt
            
            # Check for speculative decoding info
            # SGLang includes this in usage or in special fields
            verify_ct = usage.get("spec_verify_ct", 0)
            if verify_ct > 0:
                has_spec = True
                total_verify_ct += verify_ct
                sample_als.append(comp / verify_ct)
        
        metrics = {
            "latency": round(latency, 3),
            "throughput": round(total_completion_tokens / latency, 3) if latency > 0 else 0,
            "total_output_tokens": total_completion_tokens,
            "total_prompt_tokens": total_prompt_tokens,
            "has_speculative": has_spec,
        }
        
        if has_spec and sample_als:
            metrics["macro_accept_length"] = round(np.mean(sample_als), 3)
            metrics["micro_accept_length"] = round(
                total_completion_tokens / total_verify_ct, 3
            ) if total_verify_ct > 0 else 0
            metrics["min_accept_length"] = round(min(sample_als), 3)
            metrics["max_accept_length"] = round(max(sample_als), 3)
        
        return metrics
    
    def save_results(self, result, answer_file, result_file):
        """Save results in qf_mtp_eval compatible format."""
        questions = result["questions"]
        responses = result["responses"]
        
        # Save answers
        with open(answer_file, "w", encoding="utf-8") as f:
            for i, (q, resp) in enumerate(zip(questions, responses)):
                if "error" in resp:
                    continue
                answer = resp.get("choices", [{}])[0].get("message", {}).get("content", "")
                record = {
                    "question_id": q["id"],
                    "answer_id": uuid.uuid4().hex,
                    "model_id": resp.get("model", "unknown"),
                    "input": {"messages": q["messages"], "system": q.get("system", "")},
                    "choices": {"index": 0, "answer": answer},
                    "meta": resp.get("usage", {}),
                    "tstamp": time.time(),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        
        # Save result summary
        with open(result_file, "w", encoding="utf-8") as f:
            json.dump(result["metrics"], f, indent=2, ensure_ascii=False)
        
        print(f"Answers: {answer_file}")
        print(f"Results: {result_file}")


print("MTPBenchmark class defined.")
print("\nUsage:")
print('  bench = MTPBenchmark(host="127.0.0.1", port=30000)')
print('  result = bench.run("data/sample_questions.jsonl", num_questions=10)')
print('  bench.save_results(result, "answers.jsonl", "result.json")')

In [ ]:
# 运行自定义 benchmark（确保 SGLang 服务已启动）
# bench = MTPBenchmark(host="127.0.0.1", port=30000)
# result = bench.run(QUESTION_FILE, num_questions=10, max_tokens=256, num_threads=4)
# 
# print("\n=== Benchmark Result ===")
# print(json.dumps(result["metrics"], indent=2))
#
# bench.save_results(
#     result,
#     answer_file=os.path.join(DATA_DIR, "custom_answers.jsonl"),
#     result_file=os.path.join(DATA_DIR, "custom_result.json"),
# )

## 5. 构造自定义 question.jsonl

如果你想评测特定场景（如代码生成、数学推理等），可以按以下格式构造数据：

In [ ]:
# 构造自定义评测数据
custom_questions = [
    # 代码生成场景
    {
        "messages": [
            {"role": "system", "content": "You are an expert Python programmer."},
            {"role": "user", "content": "Implement a LRU cache with O(1) get and put operations."},
        ]
    },
    # 数学推理场景
    {
        "messages": [
            {"role": "system", "content": "You are a math tutor. Show your work step by step."},
            {"role": "user", "content": "Prove that sqrt(2) is irrational."},
        ]
    },
    # 多轮对话场景
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is a binary search tree?"},
            {"role": "assistant", "content": "A BST is a tree data structure where each node has at most two children, with left children smaller and right children larger than the parent."},
            {"role": "user", "content": "What is the time complexity of searching in a balanced BST?"},
        ]
    },
]

# 保存自定义数据
custom_file = os.path.join(DATA_DIR, "custom_bench.jsonl")
with open(custom_file, "w", encoding="utf-8") as f:
    for q in custom_questions:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print(f"Custom benchmark data saved to: {custom_file}")
print(f"Questions: {len(custom_questions)}")

# 验证格式
loaded = load_questions(custom_file)
mode, sc, mc = detect_turn_mode(loaded)
print(f"Mode: {mode}, Single: {sc}, Multi: {mc}")

## 本节小结

| 知识点 | 掌握内容 |
|--------|----------|
| bench_sglang_eagle.py | 基于 SDK 的批量评测, load_questions 多格式支持 |
| bench_sglang_chat.py | HTTP 并发评测, MTP Head 逐头分析 |
| 指标体系 | macro/micro AL, mtp_head_accept_rates, throughput/latency |
| 自定义 benchmark | 构造 question.jsonl, 使用 MTPBenchmark 类 |
| Thinking mode | 添加 `<think>` token 触发 CoT 推理 |

---
**下一节**: 06_full_experiment — 完整实验：端到端 MTP 评测流程